3. 建模

In [1]:
import pandas as pd

# 读取原始数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/词频数据最终文件.csv')

# 分组汇总
result = df.groupby(['year', '企业规模'])['年报全文人工智能关键词词频'].sum().reset_index()

# 重命名列
result.columns = ['年份', '企业规模', '词频总计']

# 保存新文件
result.to_csv('词频建模数据.csv', index=False, encoding='utf-8-sig')

In [2]:
import pandas as pd

# 读取两个文件
df1 = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/全国企业创新调查-主表.csv')
df2 = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/词频建模数据.csv')

# 合并
merged = pd.merge(df1, df2, on=['年份', '企业规模'], how='inner')

# 保存新文件
merged.to_csv('合并词频建模数据.csv', index=False, encoding='utf-8-sig')

In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error

# 读取数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/合并词频建模数据.csv')

# 选取自变量和因变量
X = df[['开展创新活动企业数', '创新费用支出合计(亿元)', '开展创新合作企业数',
        '采取了知识产权保护或相关措施的企业数', '制定创新战略目标的企业数']]
y = df['词频总计']

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 参数网格
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 网格搜索交叉验证
rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)

print('最佳参数:', grid_search.best_params_)
print('最佳交叉验证R2:', grid_search.best_score_)

# 用最优参数模型在测试集上评估
best_rf = grid_search.best_estimator_
y_pred_test = best_rf.predict(X_test)
print('测试集R2:', r2_score(y_test, y_pred_test))
print('测试集MSE:', mean_squared_error(y_test, y_pred_test))

# 打印特征重要性（即“系数”）
feature_names = X.columns
importances = best_rf.feature_importances_
for name, importance in zip(feature_names, importances):
    print(f"{name}: {importance:.4f}")

最佳参数: {'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
最佳交叉验证R2: 0.7853464613234209
测试集R2: 0.8149738551292446
测试集MSE: 12914008.2268125
开展创新活动企业数: 0.1057
创新费用支出合计(亿元): 0.0617
开展创新合作企业数: 0.1378
采取了知识产权保护或相关措施的企业数: 0.4481
制定创新战略目标的企业数: 0.2468


/opt/conda/lib/python3.6/site-packages/sklearn/model_selection/_search.py:813: DeprecationWarning: The default of the `iid` parameter will change from True to False in version 0.22 and will be removed in 0.24. This will change numeric results when test-set sizes are unequal.
  DeprecationWarning)


In [4]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
# 设置支持中文的字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  
plt.rcParams['axes.unicode_minus'] = False 

In [5]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 特征重要性表
importance_df = pd.DataFrame({
    '特征': feature_names,
    '重要性': np.round(importances, 3)
}).sort_values('重要性', ascending=False)

# 2. 评估指标表
metrics_df = pd.DataFrame({
    '指标': ['最佳交叉验证R2', '测试集R2', '测试集MSE'],
    '值': [
        f"{grid_search.best_score_:.3f}",
        f"{r2_score(y_test, y_pred_test):.3f}",
        f"{mean_squared_error(y_test, y_pred_test):.3f}"
    ]
})

# 绘制特征重要性表
fig1, ax1 = plt.subplots(figsize=(6, 1 + 0.5*len(importance_df)))
ax1.axis('off')
tbl1 = ax1.table(
    cellText=importance_df.values,
    colLabels=importance_df.columns,
    loc='center',
    cellLoc='center'
)
tbl1.auto_set_font_size(False)
tbl1.set_fontsize(12)
tbl1.scale(1.2, 1.2)
plt.tight_layout()
plt.show()

# 绘制评估指标表
fig2, ax2 = plt.subplots(figsize=(4, 2))
ax2.axis('off')
tbl2 = ax2.table(
    cellText=metrics_df.values,
    colLabels=metrics_df.columns,
    loc='center',
    cellLoc='center'
)
tbl2.auto_set_font_size(False)
tbl2.set_fontsize(12)
tbl2.scale(1.2, 1.2)
plt.tight_layout()
plt.show()

<Figure size 432x252 with 1 Axes>

<Figure size 288x144 with 1 Axes>